# LAB 09 — Transformer NER, Evaluation và Information Extraction

Notebook dành cho bài 14, 16, 18–27. Dùng CoNLL-2003 và `distilbert-base-cased`. Cấu hình seed là 42. Phần training chính bị khóa cho đến khi sinh viên hoàn thành `prediction.md`; có thể chạy các cell kiểm tra dữ liệu/alignment trước đó.

## 1. Setup, package versions, device

In [1]:
from pathlib import Path
import subprocess, sys
import json, csv, time, platform
import numpy as np
import pandas as pd
from IPython.display import display

# Notebook trên Colab chỉ được mở trực tiếp từ GitHub; runtime vẫn cần clone repo để có code và dữ liệu.
repo_url = "https://github.com/NguyenDucThang-tb/NLP.git"
repo_roots = [
    Path.cwd(),
    Path("/content/NLP/Lab9"),
    Path("/content/NLP/Lab09"),
    Path("/content/drive/MyDrive/NLP/Lab9"),
    Path("/content/drive/MyDrive/NLP/Lab09"),
]
LAB_DIR = next((p for p in repo_roots if (p / "ner_core.py").is_file()), None)

if LAB_DIR is None and Path("/content").exists():
    repo_dir = Path("/content/NLP")
    if not (repo_dir / "Lab9" / "ner_core.py").is_file():
        if repo_dir.exists():
            raise FileNotFoundError(
                f"Đã có {repo_dir} nhưng không thấy Lab9/ner_core.py. "
                "Xóa/đổi tên thư mục clone cũ hoặc clone repo vào thư mục khác."
            )
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(repo_dir)], check=True)
    LAB_DIR = repo_dir / "Lab9"

if LAB_DIR is None:
    raise FileNotFoundError(
        "Không tìm thấy Lab9/ner_core.py. Trong Colab, clone repository NLP trước "
        "hoặc đảm bảo toàn bộ thư mục Lab9 đã được push lên GitHub."
    )

sys.path.insert(0, str(LAB_DIR))
from ner_core import *
from transformer_ner import *
from experiment_tools import *
from transformers import AutoTokenizer
import torch

SEED = 42
DATASET_ID = DEFAULT_DATASET_ID
MODEL_ID = DEFAULT_MODEL_ID
MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
OUTPUT_DIR = LAB_DIR / "outputs"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(SEED)
print({"python": platform.python_version(), "torch": torch.__version__, "transformers": __import__("transformers").__version__, "device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})

{'python': '3.13.16', 'torch': '2.14.1+cu130', 'transformers': '5.22.1', 'device': 'cuda', 'gpu': 'Tesla T4'}


## 2. Dataset splits và label mapping

In [2]:
splits, label_names = load_conll2003(dataset_id=DATASET_ID)
label2id = {label: index for index, label in enumerate(label_names)}
id2label = {index: label for label, index in label2id.items()}
summary = pd.DataFrame(dataset_summary(splits))
display(summary)
print("Label metadata:", label_names)
print("Invalid BIO gold:", {name: len(validate_bio_sequences(part.labels)) for name, part in splits.items()})
assert all(len(part.words) == len(part.labels) for part in splits.values())
write_environment_metadata(OUTPUT_DIR / "environment.json", DATASET_ID, splits, MODEL_ID, SEED)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


,split,sentences,words,entity_counts,bio_label_counts,invalid_bio_sequences
0,train,14041,203618,"{'ORG': 6321, 'MISC': 3438, 'PER': 6600, 'LOC'...","{'B-ORG': 6321, 'O': 169575, 'B-MISC': 3438, '...",0
1,validation,3250,51360,"{'ORG': 1341, 'LOC': 1837, 'PER': 1842, 'MISC'...","{'O': 42757, 'B-ORG': 1341, 'B-LOC': 1837, 'B-...",0
2,test,3453,46427,"{'PER': 1617, 'LOC': 1668, 'MISC': 702, 'ORG':...","{'O': 38315, 'B-LOC': 1668, 'B-PER': 1617, 'I-...",0


Label metadata: ['O', 'B-ORG', 'B-MISC', 'B-PER', 'I-PER', 'B-LOC', 'I-ORG', 'I-MISC', 'I-LOC']
Invalid BIO gold: {'train': 0, 'validation': 0, 'test': 0}


{'dataset_source': 'tner/conll2003',
 'dataset_license_note': 'See the CoNLL-2003 dataset card and original Reuters data terms.',
 'split_sizes_sentences': {'train': 14041, 'validation': 3250, 'test': 3453},
 'split_sizes_words': {'train': 203618, 'validation': 51360, 'test': 46427},
 'base_model': 'distilbert/distilbert-base-cased',
 'random_seed': 42,
 'python': '3.13.16',
 'platform': 'Linux-6.6.159+-x86_64-with-glibc2.39',
 'packages': {'torch': '2.14.1+cu130',
  'transformers': '5.22.1',
  'datasets': '4.8.5',
  'accelerate': '1.15.0',
  'scikit-learn': '1.6.1',
  'seqeval': None,
  'numpy': '2.4.6',
  'pandas': '2.2.3',
  'matplotlib': '3.10.0'},
 'cuda_available': True,
 'gpu': 'Tesla T4'}

## 3. Fast tokenizer và label alignment (Strategy A)

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
assert tokenizer.is_fast
words, gold_labels = splits["train"].words[0], splits["train"].labels[0]
encoded = tokenizer(words, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH)
alignment = align_words(words, gold_labels, tokenizer, label2id, MAX_LENGTH)
print("Original words:", words)
print("Original BIO:", gold_labels)
print("Subword tokens:", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print("input_ids:", encoded["input_ids"])
print("attention_mask:", encoded["attention_mask"])
print("word_ids:", encoded.word_ids())
print("aligned labels:", [id2label[x] if x != IGNORE_INDEX else "IGNORE" for x in alignment["aligned_labels"]])
print("split words:", alignment["split_words"])
print("truncated word indices:", alignment["truncated_word_ids"])

Original words: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
Original BIO: ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']
Subword tokens: ['[CLS]', 'EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'la', '##mb', '.', '[SEP]']
input_ids: [101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
word_ids: [None, 0, 1, 2, 3, 4, 5, 6, 7, 7, 8, None]
aligned labels: ['IGNORE', 'B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'IGNORE', 'O', 'IGNORE']
split words: {7: 2}
truncated word indices: []


## 4. Encode splits and report truncation

In [4]:
encoded_splits = {name: encode_split(part, tokenizer, label2id, MAX_LENGTH) for name, part in splits.items()}
truncation_table = pd.DataFrame({name: truncation_report(rows) for name, rows in encoded_splits.items()}).T
display(truncation_table)
print("Collator will pad label positions with IGNORE_INDEX =", IGNORE_INDEX)

,sentences,original_words,evaluated_words,truncated_words,truncated_sentences,word_retention_percent
train,14041.0,203618.0,203618.0,0.0,0.0,100.0
validation,3250.0,51360.0,51360.0,0.0,0.0,100.0
test,3453.0,46427.0,46427.0,0.0,0.0,100.0


Collator will pad label positions with IGNORE_INDEX = -100


## 5. Training configuration and student prediction gate

In [ ]:
training_config = {"seed": SEED, "dataset": DATASET_ID, "model": MODEL_ID, "max_length": MAX_LENGTH,
                   "batch_size": BATCH_SIZE, "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
                   "weight_decay": WEIGHT_DECAY, "optimizer": "AdamW", "device": str(DEVICE)}
print(json.dumps(training_config, indent=2))

prediction_path = LAB_DIR / "prediction.md"
prediction_text = prediction_path.read_text(encoding="utf-8") if prediction_path.exists() else "TODO"
PREDICTION_COMPLETED, prediction_status = prediction_ready(prediction_path)
RUN_FULL_TRAINING = False  # Chỉ bật sau khi tự hoàn thành prediction.md.
print("prediction.md:", prediction_path.exists(), "| prediction completed:", PREDICTION_COMPLETED, "|", prediction_status)

{
  "seed": 42,
  "dataset": "tner/conll2003",
  "model": "distilbert/distilbert-base-cased",
  "max_length": 256,
  "batch_size": 8,
  "epochs": 3,
  "learning_rate": 2e-05,
  "weight_decay": 0.01,
  "optimizer": "AdamW",
  "device": "cuda"
}
prediction.md: True | prediction completed: True | Prediction file appears complete


## 6. Fine-tune model — best checkpoint theo validation entity F1

In [ ]:
if RUN_FULL_TRAINING and PREDICTION_COMPLETED:
    model, train_result, training_history = train_transformer(
        encoded_splits["train"], encoded_splits["validation"], tokenizer, label_names, MODEL_ID, OUTPUT_DIR,
        epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY, seed=SEED, max_length=MAX_LENGTH
    )
    print(train_result)
else:
    print("SKIPPED: hoàn thành prediction.md và đặt RUN_FULL_TRAINING=True trước training chính thức.")

SKIPPED: hoàn thành prediction.md và đặt RUN_FULL_TRAINING=True trước training chính thức.


## 7. Test evaluation — chỉ dùng sau khi đã chọn checkpoint

In [7]:
checkpoint = OUTPUT_DIR / "checkpoints" / "best"
if checkpoint.exists():
    from transformers import AutoModelForTokenClassification
    model = AutoModelForTokenClassification.from_pretrained(checkpoint).to(DEVICE)
    test_gold, test_pred = evaluate_examples(model, encoded_splits["test"], tokenizer, label_names, BATCH_SIZE, DEVICE)
    entity_types = sorted({label.split("-", 1)[-1] for label in label_names if label != "O"})
    test_metrics = evaluate_tags(test_gold, test_pred, entity_types)
    print(json.dumps({k:v for k,v in test_metrics.items() if k not in {"per_type", "length_metrics"}}, indent=2))
    save_model_predictions(splits["test"], test_gold, test_pred, OUTPUT_DIR / "transformer" / "test_predictions.jsonl", [row["kept_word_ids"] for row in encoded_splits["test"]])
else:
    print("NOT_RUN: chưa có best checkpoint; không tính hoặc ghi metrics Transformer giả.")

NOT_RUN: chưa có best checkpoint; không tính hoặc ghi metrics Transformer giả.


## 8. Experiment 2–5: metrics, per-type, entity length, confusion matrix

In [8]:
if checkpoint.exists():
    # So sánh baseline chỉ trên cùng các word positions được giữ lại bởi tokenizer.
    result_rows = []
    baseline_prediction_path = OUTPUT_DIR / "baseline" / "test_predictions.jsonl"
    if baseline_prediction_path.exists():
        baseline_records = [json.loads(line) for line in baseline_prediction_path.read_text(encoding="utf-8").splitlines()]
        kept = [row["kept_word_ids"] for row in encoded_splits["test"]]
        baseline_gold = [[splits["test"].labels[i][j] for j in kept[i]] for i in range(len(kept))]
        baseline_pred = [[baseline_records[i]["prediction"][j] for j in kept[i]] for i in range(len(kept))]
        baseline_metrics = evaluate_tags(baseline_gold, baseline_pred, entity_types)
        result_rows.append({"experiment":"baseline_ner", "model":"Token features + Logistic Regression",
            "dataset":DATASET_ID, "precision":baseline_metrics["entity_precision_micro"],
            "recall":baseline_metrics["entity_recall_micro"], "f1":baseline_metrics["entity_f1_micro"],
            "test_sentences":len(kept), "evaluated_words":sum(map(len, kept)),
            "evaluation_unit":"strict exact entity span; same retained word positions", "status":"EXECUTED"})
    result_rows.append({"experiment":"transformer_ner", "model":"Fine-tuned DistilBERT",
        "dataset":DATASET_ID, "precision":test_metrics["entity_precision_micro"],
        "recall":test_metrics["entity_recall_micro"], "f1":test_metrics["entity_f1_micro"],
        "test_sentences":len(test_gold), "evaluated_words":sum(map(len, test_gold)),
        "training_time_seconds":train_result["training_time_seconds"] if "train_result" in globals() else None,
        "checkpoint":str(checkpoint), "evaluation_unit":"strict exact entity span; same retained word positions", "status":"EXECUTED"})
    result_frame = pd.DataFrame(result_rows)
    display(result_frame)
    results_path = LAB_DIR / "results.csv"
    if results_path.exists() and results_path.stat().st_size:
        previous = pd.read_csv(results_path)
        previous = previous[~previous["model"].isin(result_frame["model"])]
        result_frame = pd.concat([previous, result_frame], ignore_index=True, sort=False)
    result_frame.to_csv(results_path, index=False)

    type_rows = [{"model":"Fine-tuned DistilBERT", "entity_type":key, **value}
                 for key,value in test_metrics["per_type"].items()]
    type_frame = pd.DataFrame(type_rows)
    type_path = LAB_DIR / "entity_metrics.csv"
    if type_path.exists() and type_path.stat().st_size:
        previous_types = pd.read_csv(type_path)
        previous_types = previous_types[previous_types["model"] != "Fine-tuned DistilBERT"]
        type_frame = pd.concat([previous_types, type_frame], ignore_index=True, sort=False)
    type_frame.to_csv(type_path, index=False)
    pd.DataFrame(test_metrics["length_metrics"]).to_csv(OUTPUT_DIR / "transformer" / "entity_length_metrics.csv", index=False)

    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    import matplotlib.pyplot as plt
    flat_gold = [x for row in test_gold for x in row]; flat_pred = [x for row in test_pred for x in row]
    cm = confusion_matrix(flat_gold, flat_pred, labels=label_names)
    fig, ax = plt.subplots(figsize=(10,8)); ConfusionMatrixDisplay(cm, display_labels=label_names).plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
    ax.set_title("Fine-tuned Transformer — token-level BIO confusion matrix"); ax.set_xlabel("Predicted label"); ax.set_ylabel("Gold label")
    fig.tight_layout(); fig.savefig(LAB_DIR / "confusion_matrix.png", dpi=180); plt.show()

    length_frame = pd.DataFrame(test_metrics["length_metrics"])
    fig, ax = plt.subplots(figsize=(7,4)); ax.bar(length_frame["entity_length"], length_frame["f1"].fillna(0))
    ax.set(title="Entity length vs NER performance", xlabel="Gold entity length (original words)", ylabel="Exact-span F1", ylim=(0,1))
    fig.tight_layout(); fig.savefig(LAB_DIR / "entity_length_analysis.png", dpi=180); plt.show()
else:
    print("NOT_RUN: no Transformer test metrics, entity table, or Transformer plots created.")

NOT_RUN: no Transformer test metrics, entity table, or Transformer plots created.


## 9. Error analysis evidence (bài 23)

In [9]:
if checkpoint.exists():
    # Tạo bằng chứng FP/FN/boundary từ test predictions; để Student explanation trống.
    error_rows = []
    for sid, (words, gold_tags, pred_tags) in enumerate(zip(splits["test"].words, test_gold, test_pred)):
        gs = extract_bio_spans(gold_tags, strict=True)
        ps = extract_bio_spans(pred_tags, strict=False)
        for entity_type, start, end in sorted(ps - gs):
            error_rows.append({"sentence_id":sid,"sentence":" ".join(words),"gold_entity":"—",
                "predicted_entity":" ".join(words[start:end]),"gold_type":"—","predicted_type":entity_type,
                "error_category":"false positive / spurious candidate","student_explanation":"TODO"})
        for entity_type, start, end in sorted(gs - ps):
            error_rows.append({"sentence_id":sid,"sentence":" ".join(words),"gold_entity":" ".join(words[start:end]),
                "predicted_entity":"—","gold_type":entity_type,"predicted_type":"—",
                "error_category":"false negative / missing candidate","student_explanation":"TODO"})
        for ge in gs:
            for pe in ps:
                if ge[0] == pe[0] and max(ge[1],pe[1]) < min(ge[2],pe[2]) and (ge[1],ge[2]) != (pe[1],pe[2]):
                    error_rows.append({"sentence_id":sid,"sentence":" ".join(words),
                        "gold_entity":" ".join(words[ge[1]:ge[2]]),"predicted_entity":" ".join(words[pe[1]:pe[2]]),
                        "gold_type":ge[0],"predicted_type":pe[0],"error_category":"boundary candidate","student_explanation":"TODO"})
                    break
    evidence = pd.DataFrame(error_rows)
    evidence.to_csv(OUTPUT_DIR / "transformer" / "error_candidates.csv", index=False)
    category_order = ["false positive / spurious candidate", "false negative / missing candidate", "boundary candidate"]
    counts = evidence.groupby("error_category").size().to_dict() if not evidence.empty else {}
    lines = ["# Error analysis — Transformer test evidence", "",
        "Các ứng viên dưới đây lấy từ test predictions. Nhãn lỗi là gợi ý theo span; sinh viên cần kiểm tra câu và tự viết explanation.", "",
        f"Candidate counts: {counts}", "", "| Sentence ID | Sentence | Gold entity/type | Predicted entity/type | Candidate category | Student explanation |",
        "|---:|---|---|---|---|---|"]
    for category in category_order:
        subset = evidence[evidence["error_category"] == category] if not evidence.empty else evidence
        for _, row in subset.head(5).iterrows():
            gold_value = f"{row['gold_entity']} / {row['gold_type']}"
            pred_value = f"{row['predicted_entity']} / {row['predicted_type']}"
            lines.append(f"| {row['sentence_id']} | {row['sentence']} | {gold_value} | {pred_value} | {category} | TODO |")
    lines += ["", "## Manual analysis", "", "TODO: Chọn ít nhất 5 FP, 5 FN và 5 boundary errors nếu số candidate thực tế cho phép; ghi evidence/offset và tự giải thích. Không suy nguyên nhân chỉ từ nhãn candidate."]
    (LAB_DIR / "error_analysis.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    display(evidence.groupby("error_category").size().rename("candidates"))
else:
    print("NOT_RUN: chờ Transformer predictions. Không tạo mẫu lỗi giả.")

NOT_RUN: chờ Transformer predictions. Không tạo mẫu lỗi giả.


## 10. Context analysis (bài 24) và Mini IE gold (bài 26–27)

In [10]:
# Same surface form/different entity types lấy trực tiếp từ gold labels của test split.
context_examples = write_context_analysis(splits["test"], LAB_DIR / "context_analysis.md")
print("Same surface / different type examples:", len(context_examples))

# 30 câu có gold spans từ CoNLL-2003 test annotations; đây không phải prediction của model.
write_ie_gold_template(splits["test"], LAB_DIR / "ie_gold_template.csv", count=30)
print("Saved CoNLL gold-backed IE sample:", LAB_DIR / "ie_gold_template.csv")

if checkpoint.exists():
    from information_extraction import predict_entities
    ie_rows = list(csv.DictReader((LAB_DIR / "ie_gold_template.csv").open(encoding="utf-8")))
    model.to(DEVICE)
    pred_records=[]; tp=fp=fn=0
    for row in ie_rows:
        result = predict_entities(row["text"], model, tokenizer, DEVICE, MAX_LENGTH)
        pred_set={(item["start"],item["end"],item["label"]) for item in result["entities"]}
        gold_set={(item["start"],item["end"],item["label"]) for item in json.loads(row["gold_entities_json"])}
        tp += len(pred_set & gold_set); fp += len(pred_set-gold_set); fn += len(gold_set-pred_set)
        pred_records.append({"sentence_id":row["sentence_id"],"prediction":result["entities"],"gold":list(gold_set)})
    ie_precision=tp/(tp+fp) if tp+fp else None; ie_recall=tp/(tp+fn) if tp+fn else None
    ie_f1=2*ie_precision*ie_recall/(ie_precision+ie_recall) if ie_precision is not None and ie_recall is not None and ie_precision+ie_recall else None
    pd.DataFrame([{"sentences":len(ie_rows),"tp":tp,"fp":fp,"fn":fn,"precision":ie_precision,"recall":ie_recall,"f1":ie_f1,"gold_source":"CoNLL-2003 test annotations","status":"EXECUTED"}]).to_csv(LAB_DIR/"ie_evaluation.csv",index=False)
    (OUTPUT_DIR/"transformer"/"ie_predictions.json").write_text(json.dumps(pred_records,ensure_ascii=False,indent=2),encoding="utf-8")
    display(pd.read_csv(LAB_DIR/"ie_evaluation.csv"))
else:
    print("IE evaluation NOT_RUN: model checkpoint chưa có; gold sample có sẵn từ CoNLL test annotations.")

Same surface / different type examples: 69
Saved CoNLL gold-backed IE sample: /content/NLP/Lab9/ie_gold_template.csv
IE evaluation NOT_RUN: model checkpoint chưa có; gold sample có sẵn từ CoNLL test annotations.
